# Telco Customer Churn Prediction and Behavior Analytics
**Student:** Vedant  
**Program:** IBM SkillsBuild Data Analytics with AI Academic Internship — BharatCares / AICTE  
**Dataset:** IBM Telco Customer Churn (`WA_Fn-UseC_-Telco-Customer-Churn.csv`)

---

## Cell 1 — Environment Setup & Library Imports

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 1 | Environment Setup & Library Imports
# ─────────────────────────────────────────────────────────────────────────────

# ── Standard library ──────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

# ── Data manipulation ─────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── Preprocessing ─────────────────────────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

# ── Models ────────────────────────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# ── Evaluation ────────────────────────────────────────────────────────────────
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    roc_curve,
    classification_report,
)

# ── Global plot style ─────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 13, 'axes.labelsize': 11})

print('✅ All libraries imported successfully.')

## Cell 2 — Dataset Loading & Initial Inspection

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 | Dataset Loading, Shape, Inspection & TotalCharges Fix
# ─────────────────────────────────────────────────────────────────────────────

DATASET_PATH = 'WA_Fn-UseC_-Telco-Customer-Churn.csv'

# ── Load ──────────────────────────────────────────────────────────────────────
df_raw = pd.read_csv(DATASET_PATH)
print(f'Dataset shape  : {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')

# ── Basic peek ────────────────────────────────────────────────────────────────
display(df_raw.head())

# ── Data types & non-null counts ──────────────────────────────────────────────
print('\n── dtypes & null counts ──')
display(df_raw.info())

# ── Descriptive statistics ────────────────────────────────────────────────────
print('\n── Descriptive statistics (numeric columns) ──')
display(df_raw.describe())

# ── Fix: TotalCharges contains spaces → convert to float, drop/fill NaN rows ──
#   The column is read as object because some rows contain a single space ' '.
df_raw['TotalCharges'] = pd.to_numeric(df_raw['TotalCharges'], errors='coerce')

n_nan = df_raw['TotalCharges'].isna().sum()
print(f'\nTotalCharges NaN after coercion: {n_nan}')

# Impute with median (all NaN rows correspond to new customers with tenure=0)
df_raw['TotalCharges'].fillna(df_raw['TotalCharges'].median(), inplace=True)

print(f'TotalCharges dtype after fix  : {df_raw["TotalCharges"].dtype}')

# ── Duplicate check ───────────────────────────────────────────────────────────
n_dup = df_raw.duplicated().sum()
print(f'Duplicate rows                : {n_dup}')

# ── Churn label distribution ──────────────────────────────────────────────────
print('\nChurn value counts:')
print(df_raw['Churn'].value_counts())
print(f'Churn rate: {df_raw["Churn"].value_counts(normalize=True)["Yes"]*100:.1f}%')

## Cell 3 — Exploratory Data Analysis (EDA)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 3 | Exploratory Data Analysis
# ─────────────────────────────────────────────────────────────────────────────

df = df_raw.copy()   # work on a copy to preserve raw data

# ── 3.1 Churn Distribution (Count + Percent) ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

churn_counts = df['Churn'].value_counts()
churn_pct    = df['Churn'].value_counts(normalize=True) * 100

axes[0].bar(churn_counts.index, churn_counts.values,
            color=['#4CAF50', '#F44336'], edgecolor='white', width=0.5)
axes[0].set_title('Churn Count Distribution')
axes[0].set_xlabel('Churn')
axes[0].set_ylabel('Number of Customers')
for bar, val in zip(axes[0].patches, churn_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 40,
                 f'{val:,}', ha='center', fontsize=11, fontweight='bold')

axes[1].pie(churn_pct.values, labels=churn_pct.index,
            autopct='%1.1f%%', colors=['#4CAF50', '#F44336'],
            startangle=140, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Churn Proportion')

plt.suptitle('Overall Churn Distribution', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# ── 3.2 Tenure vs Churn (KDE + Histogram) ────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
for label, color in zip(['No', 'Yes'], ['#4CAF50', '#F44336']):
    subset = df[df['Churn'] == label]['tenure']
    ax.hist(subset, bins=30, alpha=0.5, label=f'Churn={label}',
            color=color, edgecolor='white', density=True)
    subset.plot.kde(ax=ax, color=color, linewidth=2)
ax.set_title('Tenure Distribution by Churn Status')
ax.set_xlabel('Tenure (months)')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.show()

# ── 3.3 Monthly Charges vs Churn (Box + Strip) ────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=df, x='Churn', y='MonthlyCharges',
            palette={'No': '#4CAF50', 'Yes': '#F44336'}, width=0.4, ax=ax)
sns.stripplot(data=df, x='Churn', y='MonthlyCharges',
              palette={'No': '#4CAF50', 'Yes': '#F44336'},
              alpha=0.15, jitter=True, size=2.5, ax=ax)
ax.set_title('Monthly Charges vs Churn')
ax.set_xlabel('Churn')
ax.set_ylabel('Monthly Charges ($)')
plt.tight_layout()
plt.show()

# ── 3.4 Contract Type vs Churn Rate ──────────────────────────────────────────
contract_churn = (df.groupby(['Contract', 'Churn']).size()
                    .unstack(fill_value=0))
contract_churn['ChurnRate'] = (contract_churn['Yes'] /
                               contract_churn.sum(axis=1) * 100)
fig, ax = plt.subplots(figsize=(8, 4))
contract_churn['ChurnRate'].sort_values(ascending=False).plot(
    kind='bar', ax=ax, color=['#F44336', '#FF9800', '#4CAF50'],
    edgecolor='white', width=0.5)
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.set_title('Churn Rate by Contract Type')
ax.set_xlabel('Contract Type')
ax.set_ylabel('Churn Rate (%)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()

# ── 3.5 Correlation Heatmap (numeric features) ───────────────────────────────
num_cols = df.select_dtypes(include='number').columns.tolist()
# Encode target temporarily for correlation
temp_churn = (df['Churn'] == 'Yes').astype(int)
corr_df = df[num_cols].copy()
corr_df['Churn'] = temp_churn

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_df.corr(), annot=True, fmt='.2f',
            cmap='coolwarm', linewidths=0.5, ax=ax)
ax.set_title('Correlation Heatmap (Numeric Features + Churn)')
plt.tight_layout()
plt.show()

print('✅ EDA complete.')

## Cell 4 — Data Preprocessing

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 4 | Data Preprocessing
#  • Drop customerID (unique identifier — no predictive value)
#  • Binary categorical columns → LabelEncoder (2 unique values)
#  • Multi-class categorical columns → One-Hot Encoding
#  • Numeric features → StandardScaler
# ─────────────────────────────────────────────────────────────────────────────

df_proc = df.copy()

# ── Drop ID column ────────────────────────────────────────────────────────────
df_proc.drop(columns=['customerID'], inplace=True)

# ── Encode target variable ────────────────────────────────────────────────────
df_proc['Churn'] = (df_proc['Churn'] == 'Yes').astype(int)

# ── Identify column types ─────────────────────────────────────────────────────
cat_cols     = df_proc.select_dtypes(include='object').columns.tolist()
num_cols     = df_proc.select_dtypes(include='number').columns.tolist()
num_cols.remove('Churn')   # target — do not scale

print(f'Categorical columns ({len(cat_cols)}): {cat_cols}')
print(f'Numerical  columns  ({len(num_cols)}): {num_cols}')

# ── Binary columns → Label Encoding ──────────────────────────────────────────
binary_cols = [c for c in cat_cols
               if df_proc[c].nunique() == 2]
le = LabelEncoder()
for col in binary_cols:
    df_proc[col] = le.fit_transform(df_proc[col])
print(f'\nLabel-encoded binary columns ({len(binary_cols)}): {binary_cols}')

# ── Multi-class columns → One-Hot Encoding ───────────────────────────────────
multi_cols = [c for c in cat_cols
              if df_proc[c].nunique() > 2]
df_proc = pd.get_dummies(df_proc, columns=multi_cols, drop_first=True)
print(f'One-hot-encoded multi-class columns ({len(multi_cols)}): {multi_cols}')

# ── StandardScaler on numeric features ───────────────────────────────────────
scaler = StandardScaler()
df_proc[num_cols] = scaler.fit_transform(df_proc[num_cols])

print(f'\nFinal preprocessed shape: {df_proc.shape}')
display(df_proc.head())

## Cell 5 — Stratified Train-Test Split (80 : 20)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5 | Stratified Train-Test Split (80 : 20)
# ─────────────────────────────────────────────────────────────────────────────

X = df_proc.drop(columns=['Churn'])
y = df_proc['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y          # preserve class ratio in both splits
)

print(f'Total samples   : {len(X):,}')
print(f'Training samples: {len(X_train):,}  ({len(X_train)/len(X)*100:.0f}%)')
print(f'Test samples    : {len(X_test):,}  ({len(X_test)/len(X)*100:.0f}%)')
print(f'\nTraining churn rate : {y_train.mean()*100:.1f}%')
print(f'Test     churn rate : {y_test.mean()*100:.1f}%')
print(f'\nFeatures used       : {X.shape[1]}')

## Cell 6 — Model Training (Logistic Regression & Random Forest)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 6 | Model Training
#  • Baseline : Logistic Regression
#  • Advanced  : Random Forest Classifier
# ─────────────────────────────────────────────────────────────────────────────

# ── Logistic Regression (Baseline) ───────────────────────────────────────────
lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight='balanced',   # handles class imbalance
    solver='lbfgs'
)
lr_model.fit(X_train, y_train)
print('✅ Logistic Regression trained.')

# ── Random Forest (Advanced) ──────────────────────────────────────────────────
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight='balanced',   # handles class imbalance
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)
print('✅ Random Forest Classifier trained.')

# ── Predictions ───────────────────────────────────────────────────────────────
lr_preds      = lr_model.predict(X_test)
lr_proba      = lr_model.predict_proba(X_test)[:, 1]

rf_preds      = rf_model.predict(X_test)
rf_proba      = rf_model.predict_proba(X_test)[:, 1]

print('\nPredictions generated for both models.')

## Cell 7 — Performance Evaluation & Comparison

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 7 | Performance Evaluation & Comparison
#  • Accuracy, Precision, Recall, F1, ROC-AUC
#  • Confusion Matrix (side-by-side)
#  • ROC Curve overlay
# ─────────────────────────────────────────────────────────────────────────────

def compute_metrics(name, y_true, y_pred, y_prob):
    """Return a dict of evaluation metrics."""
    return {
        'Model'    : name,
        'Accuracy' : round(accuracy_score(y_true, y_pred)  * 100, 2),
        'Precision': round(precision_score(y_true, y_pred) * 100, 2),
        'Recall'   : round(recall_score(y_true, y_pred)    * 100, 2),
        'F1-Score' : round(f1_score(y_true, y_pred)        * 100, 2),
        'ROC-AUC'  : round(roc_auc_score(y_true, y_prob)   * 100, 2),
    }

lr_metrics = compute_metrics('Logistic Regression', y_test, lr_preds, lr_proba)
rf_metrics = compute_metrics('Random Forest',       y_test, rf_preds, rf_proba)

results_df = pd.DataFrame([lr_metrics, rf_metrics]).set_index('Model')
print('═' * 60)
print('          MODEL PERFORMANCE COMPARISON (%)         ')
print('═' * 60)
display(results_df)

# ── Detailed Classification Reports ──────────────────────────────────────────
print('\n── Logistic Regression — Classification Report ──')
print(classification_report(y_test, lr_preds, target_names=['No Churn', 'Churn']))

print('── Random Forest — Classification Report ──')
print(classification_report(y_test, rf_preds, target_names=['No Churn', 'Churn']))

# ── Confusion Matrices ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, preds, title in zip(
    axes,
    [lr_preds, rf_preds],
    ['Logistic Regression', 'Random Forest']
):
    cm = confusion_matrix(y_test, preds)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                  display_labels=['No Churn', 'Churn'])
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f'Confusion Matrix — {title}', fontsize=12, fontweight='bold')

plt.suptitle('Confusion Matrices', fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

# ── ROC-AUC Curves ────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))

for name, proba, color in [
    ('Logistic Regression', lr_proba, '#3b82d4'),
    ('Random Forest',       rf_proba, '#F44336'),
]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc_val      = roc_auc_score(y_test, proba)
    ax.plot(fpr, tpr, lw=2, color=color,
            label=f'{name}  (AUC = {auc_val:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1.2, label='Random Classifier (AUC = 0.500)')
ax.fill_between(fpr, tpr, alpha=0.05, color='#3b82d4')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC-AUC Curve Comparison', fontsize=13, fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

## Cell 8 — Feature Importance Analysis

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 8 | Feature Importance Analysis
#  • Random Forest — Gini Impurity-based feature importances (Top 15)
#  • Logistic Regression — Absolute coefficient magnitudes (Top 15)
# ─────────────────────────────────────────────────────────────────────────────

TOP_N = 15
feature_names = X.columns.tolist()

# ── Random Forest Feature Importances ────────────────────────────────────────
rf_importances = pd.Series(
    rf_model.feature_importances_, index=feature_names
).sort_values(ascending=False)

top_rf = rf_importances.head(TOP_N)

fig, ax = plt.subplots(figsize=(10, 6))
colors = sns.color_palette('RdYlGn_r', TOP_N)
top_rf.sort_values().plot(
    kind='barh', ax=ax, color=colors, edgecolor='white'
)
ax.set_title(f'Random Forest — Top {TOP_N} Feature Importances',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score (Gini)')
ax.set_ylabel('Feature')
for bar in ax.patches:
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f'{bar.get_width():.4f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

# ── Logistic Regression Coefficient Magnitudes ───────────────────────────────
lr_coefs = pd.Series(
    np.abs(lr_model.coef_[0]), index=feature_names
).sort_values(ascending=False)

top_lr = lr_coefs.head(TOP_N)

fig, ax = plt.subplots(figsize=(10, 6))
colors_lr = sns.color_palette('Blues_r', TOP_N)
top_lr.sort_values().plot(
    kind='barh', ax=ax, color=colors_lr, edgecolor='white'
)
ax.set_title(f'Logistic Regression — Top {TOP_N} Feature Coefficients (|coef|)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('|Coefficient| Magnitude')
ax.set_ylabel('Feature')
for bar in ax.patches:
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'{bar.get_width():.4f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

# ── Insight Summary ───────────────────────────────────────────────────────────
print('═' * 62)
print(f'  TOP {TOP_N} CHURN DRIVERS — RANDOM FOREST  ')
print('═' * 62)
for rank, (feat, score) in enumerate(top_rf.items(), 1):
    print(f'  {rank:>2}. {feat:<40} {score:.4f}')

print('\n── Key Insights ──────────────────────────────────────────────')
print("""
  1. TENURE           — Short-tenure customers churn most; loyalty builds retention.
  2. MONTHLY CHARGES  — Higher monthly bills correlate strongly with churn.
  3. TOTAL CHARGES    — Indirect proxy for tenure; inverse relationship with churn.
  4. CONTRACT TYPE    — Month-to-month customers are the highest risk segment.
  5. INTERNET SERVICE — Fibre optic customers show elevated churn vs DSL.
  6. TECH SUPPORT     — Absence of tech support increases churn likelihood.
  7. ONLINE SECURITY  — Customers without online security churn more.
""")
print('✅ Feature Importance Analysis complete.')